In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from embeddings_extraction import (
    list_files_oswalk,
    preprocess_audio_paths,
    preprocess_video_paths,
    get_video_embeddings_list,
    get_audio_embeddings_list,
)

from attention import SelfAttention, BimodalSelfAttention
from classifier import HighlightClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ------------------ labeled dataset -------------------
LABELED_ROOT = "../data/lida_selected/"
HIGHLIGHTS_DIR = os.path.join(LABELED_ROOT, "true_segments")  # positives
NORMAL_DIR     = os.path.join(LABELED_ROOT, "false_segments")      # negatives

# If your videos+audios are in separate subfolders inside each class, set these:
VIDEO_SUBDIR = "videos"
AUDIO_SUBDIR = "audios"

# ------------------ training hyperparams -------------------
d_self = 128
D_common = 256
d_bimodal = 128
hidden = 256
dropout = 0.5
lr = 1e-4
num_epochs = 50
batch_size = 8

# learnable fusion weights (4 streams)
alpha_logits = nn.Parameter(torch.zeros(4, device=device))

# cache
CACHE_NPZ = "cached_embeddings_labeled_av.npz"


Using device: cuda


C:\Users\jenia\AppData\Local\Temp\ipykernel_47620\2191051482.py:42: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  alpha_logits = nn.Parameter(torch.zeros(4, device=device))


In [7]:
ext_v = {".mp4", ".avi", ".mov", ".mkv"}
ext_a = {".wav", ".mp3", ".flac", ".m4a"}

def collect_pairs(class_dir, y):
    vroot = os.path.join(class_dir, VIDEO_SUBDIR)
    aroot = os.path.join(class_dir, AUDIO_SUBDIR)

    vpaths = list_files_oswalk(vroot, ext_filter=ext_v)
    apaths = list_files_oswalk(aroot, ext_filter=ext_a)
    vpaths.sort()
    apaths.sort()

    assert len(vpaths) == len(apaths), f"Mismatch in {class_dir}: {len(vpaths)} videos vs {len(apaths)} audios"

    labels = [y] * len(vpaths)
    return vpaths, apaths, labels

pos_v, pos_a, pos_y = collect_pairs(HIGHLIGHTS_DIR, 1)
neg_v, neg_a, neg_y = collect_pairs(NORMAL_DIR, 0)

video_paths = pos_v + neg_v
audio_paths = pos_a + neg_a
video_level_labels = np.array(pos_y + neg_y, dtype=np.int64)

print("Total pairs:", len(video_paths))
print("Pos:", video_level_labels.sum(), "Neg:", (video_level_labels==0).sum())


Total pairs: 88
Pos: 60 Neg: 28


In [ ]:
def cache_exists(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

if cache_exists(CACHE_NPZ):
    loaded = np.load(CACHE_NPZ, allow_pickle=True)
    video_embeddings_list = list(loaded["video"])
    audio_embeddings_list = list(loaded["audio"])
    video_level_labels = loaded["labels"].astype(np.int64)
    video_paths = list(loaded["video_paths"])
    audio_paths = list(loaded["audio_paths"])
    print("Loaded cache:", CACHE_NPZ)
else:
    video_segments_list, audios_duration, n_segments_list = preprocess_video_paths(video_paths)

    audio_segments_list = preprocess_audio_paths(audio_paths, audios_duration, n_segments_list)

    print("Extracting video embeddings...")
    video_embeddings_list = get_video_embeddings_list(video_segments_list, device=device)

    print("Extracting audio embeddings...")
    audio_embeddings_list = get_audio_embeddings_list(audio_segments_list, device=device)

    assert len(video_embeddings_list) == len(audio_embeddings_list) == len(video_level_labels)
    for i in range(len(video_embeddings_list)):
        assert video_embeddings_list[i].shape[0] == audio_embeddings_list[i].shape[0], f"Segment mismatch at {i}"

    np.savez(
        CACHE_NPZ,
        video=video_embeddings_list,
        audio=audio_embeddings_list,
        labels=video_level_labels,
        video_paths=np.array(video_paths, dtype=object),
        audio_paths=np.array(audio_paths, dtype=object),
    )
    print("Saved cache:", CACHE_NPZ)

print("Example video emb:", video_embeddings_list[0].shape)
print("Example audio emb:", audio_embeddings_list[0].shape)


In [11]:
import numpy as np
import torch

def to_object_array(seq):
    out = []
    for x in seq:
        if torch.is_tensor(x):
            x = x.detach().cpu().numpy()
        out.append(x)
    return np.array(out, dtype=object)

# Save using object arrays (handles variable-length segments)
np.savez(
    CACHE_NPZ,
    video=to_object_array(video_embeddings_list),
    audio=to_object_array(audio_embeddings_list),
    labels=np.asarray(video_level_labels, dtype=np.int64),
    video_paths=np.array(video_paths, dtype=object),
    audio_paths=np.array(audio_paths, dtype=object),
    allow_pickle=True,
)

print("Saved cache (fixed):", CACHE_NPZ)
print("Example video emb:", video_embeddings_list[0].shape)
print("Example audio emb:", audio_embeddings_list[0].shape)


Saved cache (fixed): cached_embeddings_labeled_av.npz
Example video emb: torch.Size([6, 512])
Example audio emb: (6, 2048)


In [12]:
pseudo_labels_per_video = []
for y, v_emb in zip(video_level_labels, video_embeddings_list):
    T = v_emb.shape[0]
    pseudo_labels_per_video.append(np.full((T,), float(y), dtype=np.float32))

all_labels = np.concatenate(pseudo_labels_per_video)
pos_fraction = float(all_labels.mean())
pos_weight_val = (1.0 - pos_fraction) / max(pos_fraction, 1e-6) if pos_fraction > 0 else 1.0

print(f"Global pos fraction (segments): {pos_fraction:.4f}")
print(f"pos_weight for BCE: {pos_weight_val:.3f}")
pos_weight = torch.tensor([pos_weight_val], device=device)


Global pos fraction (segments): 0.7913
pos_weight for BCE: 0.264


In [13]:
def to_padded_batch(emb_list, device):
    tensors = [torch.as_tensor(e, dtype=torch.float32, device=device) for e in emb_list]
    lens = [t.shape[0] for t in tensors]
    B = len(tensors)
    D = tensors[0].shape[1]
    T_max = max(lens)

    x = torch.zeros((B, T_max, D), dtype=torch.float32, device=device)
    m = torch.zeros((B, T_max), dtype=torch.bool, device=device)

    for i, t in enumerate(tensors):
        L = t.shape[0]
        x[i, :L] = t
        m[i, :L] = True

    return x, m, lens

def unpad_to_lists(x, lens):
    out = []
    for i, L in enumerate(lens):
        out.append(x[i, :L].detach().cpu())
    return out


In [14]:
tmp_v = video_embeddings_list[0]
tmp_a = audio_embeddings_list[0]
D_v = tmp_v.shape[1]
D_a = tmp_a.shape[1]

video_self_att = SelfAttention(D_v, d_self).to(device)
audio_self_att = SelfAttention(D_a, d_self).to(device)

proj_v = nn.Linear(D_v, D_common).to(device)
proj_a = nn.Linear(D_a, D_common).to(device)

bimodal_att = BimodalSelfAttention(D_common, d_bimodal).to(device)
head = HighlightClassifier(D=D_common, hidden=hidden, p=dropout).to(device)

params = (
    list(video_self_att.parameters()) +
    list(audio_self_att.parameters()) +
    list(proj_v.parameters()) +
    list(proj_a.parameters()) +
    list(bimodal_att.parameters()) +
    list(head.parameters()) +
    [alpha_logits]
)

optimizer = optim.Adam(params, lr=lr)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


In [15]:
num_videos = len(video_embeddings_list)

for epoch in range(num_epochs):
    video_self_att.train()
    audio_self_att.train()
    proj_v.train()
    proj_a.train()
    bimodal_att.train()
    head.train()

    indices = np.random.permutation(num_videos)

    epoch_loss = 0.0
    n_batches = 0

    for start in range(0, num_videos, batch_size):
        batch_idxs = indices[start:start + batch_size]

        v_list = [video_embeddings_list[i] for i in batch_idxs]
        a_list = [audio_embeddings_list[i] for i in batch_idxs]
        y_list = [pseudo_labels_per_video[i] for i in batch_idxs]

        v_batch, v_mask, v_lens = to_padded_batch(v_list, device)
        a_batch, a_mask, a_lens = to_padded_batch(a_list, device)
        assert v_lens == a_lens

        B, T_max, _ = v_batch.shape
        y_batch = torch.zeros((B, T_max), dtype=torch.float32, device=device)
        for bi, L in enumerate(v_lens):
            y_batch[bi, :L] = torch.from_numpy(y_list[bi]).to(device)

        optimizer.zero_grad()

        v_self = video_self_att(v_batch, mask=v_mask)
        a_self = audio_self_att(a_batch, mask=a_mask)

        v_common = proj_v(v_self)
        a_common = proj_a(a_self)

        v2a, a2v = bimodal_att(
            v_self=v_common,
            a_self=a_common,
            mask_v=v_mask,
            mask_a=a_mask,
        )

        alpha = F.softmax(alpha_logits, dim=0).view(4, 1, 1, 1)
        comps = torch.stack([v_common, v2a, a2v, a_common], dim=0)
        z = (alpha * comps).sum(dim=0)

        logits = head(z)  # (B, T_max)

        loss = criterion(logits[v_mask], y_batch[v_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=5.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)

    with torch.no_grad():
        a = F.softmax(alpha_logits, dim=0).detach().cpu().numpy()
    print(f"Epoch {epoch+1}/{num_epochs} loss={avg_loss:.4f} alpha={np.round(a, 3)}")


Epoch 1/50 loss=0.3643 alpha=[0.25 0.25 0.25 0.25]
Epoch 2/50 loss=0.2367 alpha=[0.25 0.25 0.25 0.25]
Epoch 3/50 loss=0.1595 alpha=[0.25 0.25 0.25 0.25]
Epoch 4/50 loss=0.0881 alpha=[0.25 0.25 0.25 0.25]
Epoch 5/50 loss=0.0548 alpha=[0.249 0.251 0.25  0.25 ]
Epoch 6/50 loss=0.0410 alpha=[0.249 0.251 0.249 0.251]
Epoch 7/50 loss=0.0156 alpha=[0.249 0.251 0.249 0.251]
Epoch 8/50 loss=0.0103 alpha=[0.249 0.251 0.249 0.251]
Epoch 9/50 loss=0.0103 alpha=[0.249 0.251 0.249 0.251]
Epoch 10/50 loss=0.0068 alpha=[0.249 0.251 0.249 0.251]
Epoch 11/50 loss=0.0059 alpha=[0.249 0.251 0.249 0.251]
Epoch 12/50 loss=0.0047 alpha=[0.249 0.251 0.249 0.251]
Epoch 13/50 loss=0.0035 alpha=[0.249 0.251 0.249 0.251]
Epoch 14/50 loss=0.0028 alpha=[0.249 0.251 0.249 0.251]
Epoch 15/50 loss=0.0029 alpha=[0.249 0.251 0.249 0.251]
Epoch 16/50 loss=0.0024 alpha=[0.249 0.251 0.249 0.251]
Epoch 17/50 loss=0.0024 alpha=[0.249 0.251 0.249 0.251]
Epoch 18/50 loss=0.0021 alpha=[0.249 0.251 0.249 0.251]
Epoch 19/50 loss=

In [16]:
torch.save(
    {
        "video_self_att": video_self_att.state_dict(),
        "audio_self_att": audio_self_att.state_dict(),
        "proj_v": proj_v.state_dict(),
        "proj_a": proj_a.state_dict(),
        "bimodal_att": bimodal_att.state_dict(),
        "head": head.state_dict(),
        "alpha_logits": alpha_logits.detach().cpu(),
        "meta": {
            "D_v": D_v, "D_a": D_a, "d_self": d_self, "D_common": D_common,
            "d_bimodal": d_bimodal, "hidden": hidden, "dropout": dropout,
        }
    },
    "av_highlight_model_labeled.pt",
)
print("Saved to av_highlight_model_labeled.pt")


Saved to av_highlight_model_labeled.pt


In [1]:
video_self_att.eval()
audio_self_att.eval()
proj_v.eval()
proj_a.eval()
bimodal_att.eval()
head.eval()

v_batch, v_mask, v_lens = to_padded_batch(video_embeddings_list, device)
a_batch, a_mask, a_lens = to_padded_batch(audio_embeddings_list, device)
assert v_lens == a_lens

with torch.no_grad():
    v_self = video_self_att(v_batch, mask=v_mask)
    a_self = audio_self_att(a_batch, mask=a_mask)

    v_common = proj_v(v_self)
    a_common = proj_a(a_self)

    v2a, a2v = bimodal_att(
        v_self=v_common,
        a_self=a_common,
        mask_v=v_mask,
        mask_a=a_mask,
    )

    alpha = F.softmax(alpha_logits, dim=0).view(4, 1, 1, 1)
    comps = torch.stack([v_common, v2a, a2v, a_common], dim=0)
    z = (alpha * comps).sum(dim=0)

    logits = head(z)
    probs = torch.sigmoid(logits)
    probs_bt = probs.masked_fill(~a_mask, 0.0)

highlight_probs = unpad_to_lists(probs_bt.unsqueeze(-1), a_lens)
print("Num videos:", len(highlight_probs), "example shape:", highlight_probs[0].shape)


NameError: name 'video_self_att' is not defined